# CrackProof: QLoRA Fine-Tuning on Google Colab (Free T4 GPU)

**Objective**: Fine-tune an open-source LLM (`Qwen/Qwen2.5-3B-Instruct`) using **4-bit QLoRA (PEFT)** to act as the specialized **CrackProof Technical Interview Evaluator**.

### Why Fine-Tune for Technical Interviews?
1. **SOLO Taxonomy Rubric**: General LLMs give generic chat advice. Fine-tuning forces structured evaluation on 4 depth dimensions: `FUNDAMENTAL`, `REASONING`, `APPLICATION`, `EDGE_CASE`.
2. **Strict JSON Schema**: Guarantees deterministic JSON output with `verdict`, `correctness_score`, `depth_score`, and `missing_core_concepts`.
3. **RAG Grounding**: Model learns to cite numbered reference passages `[1]`, `[2]` rather than hallucinating facts.

## Step 1: Check NVIDIA GPU & Install Dependencies
Ensure your Colab Runtime is set to **T4 GPU** (`Runtime` -> `Change runtime type` -> `T4 GPU`).

In [ ]:
# 1. Verify NVIDIA T4 GPU is active
!nvidia-smi

# 2. Install modern ML fine-tuning stack
!pip install -q torch transformers datasets peft trl bitsandbytes accelerate
print('\n[OK] Dependencies installed successfully!')

## Step 2: Build Self-Contained Grounded Dataset
This cell generates ChatML samples covering **Java, OOP, DBMS, OS, CN, and DSA**.

In [ ]:
import json
import os

raw_training_data = [
    {
        'question': 'What is the difference between String, StringBuilder, and StringBuffer in Java?',
        'answer': 'String is immutable in Java and stored in the String Pool. StringBuilder is mutable and not thread-safe so it is fast. StringBuffer is mutable and thread-safe because its methods are synchronized.',
        'reference': '[1] From Java Specification: String is immutable; modifications create new objects. StringBuilder is unsynchronized for single-thread speed; StringBuffer is thread-safe.',
        'evaluation': {
            'verdict': 'CORRECT',
            'correctness_score': 10,
            'depth_score': 9,
            'correct_points': ['String is immutable', 'StringBuilder is mutable and unsynchronized', 'StringBuffer is thread-safe'],
            'missing_core_concepts': [],
            'misconceptions': [],
            'evidence': [
                {'evidence_type': 'FUNDAMENTAL', 'status': 'DEMONSTRATED', 'evidence_from_answer': 'Accurately distinguished immutability and thread safety across all three classes.'},
                {'evidence_type': 'REASONING', 'status': 'DEMONSTRATED', 'evidence_from_answer': 'Connected synchronization overhead to performance differences.'}
            ],
            'citations': [{'claim': 'String is immutable', 'source_number': 1}],
            'reasoning': 'Candidate gave complete and technically accurate distinction for all 3 classes.'
        }
    },
    {
        'question': 'Why is it mandatory to override hashCode() whenever you override equals() in Java?',
        'answer': 'Equal objects must have identical hashCodes. If you only override equals, HashMap will put them into different buckets and get() will fail to find an existing key.',
        'reference': '[1] From Object Contract: If two objects are equal according to equals(Object), calling hashCode() on each must produce the same integer result.',
        'evaluation': {
            'verdict': 'CORRECT',
            'correctness_score': 10,
            'depth_score': 9,
            'correct_points': ['Equal objects must return the same hashCode', 'Violating it breaks HashMap bucket lookup'],
            'missing_core_concepts': [],
            'misconceptions': [],
            'evidence': [
                {'evidence_type': 'FUNDAMENTAL', 'status': 'DEMONSTRATED', 'evidence_from_answer': 'Correctly stated the equals-hashCode contract.'},
                {'evidence_type': 'APPLICATION', 'status': 'DEMONSTRATED', 'evidence_from_answer': 'Demonstrated practical consequence on HashMap bucket routing.'}
            ],
            'citations': [{'claim': 'Equal objects must return the same hashCode', 'source_number': 1}],
            'reasoning': 'Accurate definition of contract and clear data structure impact.'
        }
    },
    {
        'question': 'What are the four main principles of object-oriented programming?',
        'answer': 'Encapsulation binds data and methods together hiding state. Inheritance lets subclasses reuse parent methods. Polymorphism lets different objects respond to the same call. Abstraction exposes only essential details.',
        'reference': '[1] From Object-Oriented Software Design: The four foundational concepts are encapsulation, inheritance, polymorphism, and abstraction.',
        'evaluation': {
            'verdict': 'CORRECT',
            'correctness_score': 10,
            'depth_score': 8,
            'correct_points': ['Encapsulation', 'Inheritance', 'Polymorphism', 'Abstraction'],
            'missing_core_concepts': [],
            'misconceptions': [],
            'evidence': [
                {'evidence_type': 'FUNDAMENTAL', 'status': 'DEMONSTRATED', 'evidence_from_answer': 'Candidate correctly identified and described all four principles.'}
            ],
            'citations': [{'claim': 'The four foundational concepts are encapsulation, inheritance, polymorphism, and abstraction', 'source_number': 1}],
            'reasoning': 'Fully correct fundamental recall with concise definitions.'
        }
    },
    {
        'question': 'What are the ACID properties in database transactions?',
        'answer': 'Atomicity means all-or-nothing. Consistency maintains database schema rules and constraints. Isolation prevents concurrent transactions from seeing uncommitted changes. Durability ensures committed data is saved even on power crash.',
        'reference': '[1] Relational Database Management Systems: Transactions must satisfy Atomicity, Consistency, Isolation, and Durability to guarantee data validity.',
        'evaluation': {
            'verdict': 'CORRECT',
            'correctness_score': 10,
            'depth_score': 9,
            'correct_points': ['Atomicity: all-or-nothing', 'Consistency: constraints preserved', 'Isolation: concurrent execution independence', 'Durability: persistence across crashes'],
            'missing_core_concepts': [],
            'misconceptions': [],
            'evidence': [
                {'evidence_type': 'FUNDAMENTAL', 'status': 'DEMONSTRATED', 'evidence_from_answer': 'Accurately defined all four ACID components.'}
            ],
            'citations': [{'claim': 'Transactions must satisfy Atomicity, Consistency, Isolation, and Durability', 'source_number': 1}],
            'reasoning': 'Comprehensive and accurate answer.'
        }
    },
    {
        'question': 'What is the fundamental difference between a Process and a Thread?',
        'answer': 'A process has its own private virtual memory space, whereas threads within a process share the same heap, code, and global data, but each thread has its own execution stack and program counter.',
        'reference': '[1] Operating System Concepts: A thread is a basic unit of CPU utilization; it shares code, data, and OS resources with peer threads but has its own stack.',
        'evaluation': {
            'verdict': 'CORRECT',
            'correctness_score': 10,
            'depth_score': 9,
            'correct_points': ['Process has separate address space', 'Threads share heap/code but have private stack and registers'],
            'missing_core_concepts': [],
            'misconceptions': [],
            'evidence': [
                {'evidence_type': 'FUNDAMENTAL', 'status': 'DEMONSTRATED', 'evidence_from_answer': 'Accurately described memory boundaries between processes and threads.'}
            ],
            'citations': [{'claim': 'shares code, data, and OS resources with peer threads but has its own stack', 'source_number': 1}],
            'reasoning': 'Spot-on explanation of process vs thread memory model.'
        }
    },
    {
        'question': 'What is the difference between TCP and UDP, and how does the 3-way handshake work?',
        'answer': 'TCP is connection-oriented, reliable, and guarantees in-order delivery via acknowledgments. UDP is connectionless and best-effort. Handshake: Client sends SYN, Server replies SYN-ACK, Client sends ACK.',
        'reference': '[1] Computer Networks: TCP establishes connections using SYN, SYN-ACK, ACK. UDP transmits datagrams without handshake.',
        'evaluation': {
            'verdict': 'CORRECT',
            'correctness_score': 10,
            'depth_score': 9,
            'correct_points': ['TCP is connection-oriented and reliable', 'UDP is connectionless', '3-way handshake: SYN -> SYN-ACK -> ACK'],
            'missing_core_concepts': [],
            'misconceptions': [],
            'evidence': [
                {'evidence_type': 'FUNDAMENTAL', 'status': 'DEMONSTRATED', 'evidence_from_answer': 'Accurately compared reliability and handshake steps.'}
            ],
            'citations': [{'claim': 'TCP establishes connections using SYN, SYN-ACK, ACK', 'source_number': 1}],
            'reasoning': 'Clear and complete explanation.'
        }
    },
    {
        'question': 'Can you run Binary Search on a singly linked list in O(log n) time?',
        'answer': 'Yes, because the linked list elements are sorted, so we can just find the middle element and recurse in O(log n) time.',
        'reference': '[1] Algorithms: Binary search requires random access O(1) to find the middle element. Singly linked lists require O(n) traversal to locate the median, degrading total search time to O(n).',
        'evaluation': {
            'verdict': 'INCORRECT',
            'correctness_score': 2,
            'depth_score': 2,
            'correct_points': ['Recognized binary search requires sorted data'],
            'missing_core_concepts': ['Linked lists lack constant time indexing', 'Finding middle takes O(n) traversal'],
            'misconceptions': ['Claiming binary search runs in O(log n) on a singly linked list'],
            'evidence': [
                {'evidence_type': 'FUNDAMENTAL', 'status': 'PARTIALLY_DEMONSTRATED', 'evidence_from_answer': 'Candidate understands binary search halves data, but failed to realize pointer traversal negates O(log n).'}
            ],
            'citations': [{'claim': 'Singly linked lists require O(n) traversal to locate the median, degrading total search time to O(n)', 'source_number': 1}],
            'reasoning': 'Major algorithmic misconception regarding pointer traversal vs random access.'
        }
    }
]

SYSTEM_PROMPT = '''You are a strict technical interviewer.
Evaluate the candidate\'s TECHNICAL UNDERSTANDING based on:
1. Technical correctness (0-10)
2. Conceptual depth (0-10)
3. SOLO Taxonomy Depth Dimensions: FUNDAMENTAL, REASONING, APPLICATION, EDGE_CASE
4. Missing core concepts & technical misconceptions
5. Grounding: Cite reference passages [1], [2] when provided.
Do NOT penalize grammar, English fluency, accent, or Hindi-English code-switching.
Always output valid JSON conforming to the CrackProof AnswerEvaluation schema.'''

os.makedirs('dataset', exist_ok=True)
formatted_samples = []
for item in raw_training_data:
    user_text = f"INTERVIEW QUESTION:\n{item['question']}\n\nCANDIDATE ANSWER:\n{item['answer']}\n\nREFERENCE MATERIAL:\n{item['reference']}"
    assistant_text = json.dumps(item['evaluation'], indent=2)
    formatted_samples.append({
        'messages': [
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user', 'content': user_text},
            {'role': 'assistant', 'content': assistant_text}
        ]
    })

with open('dataset/train.jsonl', 'w') as f:
    for s in formatted_samples[:5]:
        f.write(json.dumps(s) + '\n')

with open('dataset/val.jsonl', 'w') as f:
    for s in formatted_samples[5:]:
        f.write(json.dumps(s) + '\n')

print(f'[OK] Generated {len(formatted_samples)} ChatML evaluation samples!')

## Step 3: Load 4-Bit Quantized Base Model
We load `Qwen/Qwen2.5-3B-Instruct` using `BitsAndBytesConfig` (4-bit NF4). This fits comfortably in ~3.5 GB VRAM on the Colab T4 GPU.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_ID = 'Qwen/Qwen2.5-3B-Instruct'
print(f'Loading {MODEL_ID} in 4-bit NF4 Quantization...')

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True
)

print(f'[OK] Model successfully loaded on GPU! Allocated VRAM: {torch.cuda.memory_allocated() / 1024**3:.2f} GB')

## Step 4: Attach LoRA Adapter (PEFT)
We freeze base parameters and attach LoRA adapter matrices ($r=16, \alpha=32$) on attention and MLP projections.

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM'
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

## Step 5: Execute QLoRA Training with SFTTrainer
Runs 3 epochs on the T4 GPU in ~8-10 minutes.

In [ ]:
from datasets import load_dataset
from trl import SFTTrainer
from transformers import TrainingArguments

dataset = load_dataset('json', data_files={
    'train': 'dataset/train.jsonl',
    'validation': 'dataset/val.jsonl'
})

training_args = TrainingArguments(
    output_dir='./crackproof_qlora_adapter',
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=5,
    optim='paged_adamw_8bit',
    evaluation_strategy='steps',
    eval_steps=10,
    save_strategy='epoch',
    warmup_ratio=0.1,
    report_to='none'
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset['train'],
    eval_dataset=dataset['val'],
    peft_config=peft_config,
    max_seq_length=1536,
    tokenizer=tokenizer,
    args=training_args
)

print('Starting QLoRA Fine-Tuning on T4 GPU...\n')
trainer.train()

# Save trained adapter
trainer.model.save_pretrained('./crackproof_qlora_adapter')
tokenizer.save_pretrained('./crackproof_qlora_adapter')
print('\n[SUCCESS] Fine-tuning complete! Adapter saved to ./crackproof_qlora_adapter')

## Step 6: Benchmark Test (Base Model vs Fine-Tuned Model)
We test an authentic candidate answer to verify that the model produces structured JSON with the 4 SOLO dimensions and RAG citations.

In [ ]:
test_prompt = '''INTERVIEW QUESTION:
What is the difference between == and .equals() in Java, and what happens if you call .equals on null?

CANDIDATE ANSWER:
== checks memory reference while .equals checks value equality. But calling .equals on a null reference throws a NullPointerException.

REFERENCE MATERIAL:
[1] From Java Specification: The == operator tests reference identity. The equals() method tests equivalence relation. Invoking a method on a null reference throws NullPointerException.'''

messages = [
    {'role': 'system', 'content': SYSTEM_PROMPT},
    {'role': 'user', 'content': test_prompt}
]

formatted_input = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(formatted_input, return_tensors='pt').to('cuda')

print('Generating evaluation from Fine-Tuned CrackProof Model...\n' + '='*60)
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=600,
        temperature=0.1,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
print(response)
print('='*60)
print('\n[SUCCESS] Notice how the model directly outputs structured JSON with 4 depth dimensions and verified citations!')